# Assumption

The README did not specify if we are to use the SMTLib distribution for Z3. I am hence assuming that we are to use the python API [(documentation)](https://ericpony.github.io/z3py-tutorial/guide-examples.htm).

For me, it is easier to parse and has a cleaner syntax. I also used it for Assignment 1 and so have some level of familiarity with it. Python with jupyter notebooks also allows for easier writeups with code.



### Imports

In [ ]:
from z3 import Solver, Bool, And, Not, Or, Ints, Real, Reals, sat
import time
from itertools import combinations
from IPython.display import Markdown
import numpy as np

### **Game of 21**

We can recursively understand the logic behind the model. 

- Firstly, note that a player who is left with a pile of size 1, 2, or 3, can always win.
    - Thus, $Win(1) = Win(2) = Win(3) = \top$
        - This is the base case, kind of like axioms

- Next, note that if a player has a winning position, it means that there is some removal choice they can make so that the resulting configation is not a winning postion. In other words, having a pile of size either 1 object, 2 objects or 3 objects lesser than the current pile size that leads to a loss.
    - So, $Win(i) = \neg Win(i-1) \lor \neg Win(i-2) \lor \neg Win(i-3)$
        - This is the recursive description, kind of like a rule of inference
        - $1 < n \leq 21$
        - The winning strategy would be to always pick this removal choice.

In [2]:
s = Solver()
Win = [Bool(f"Win({i})") for i in range(1,22)]

In [3]:
# base case
s.add(Win[0]==True, Win[1]==True, Win[2]==True)
# indexing is weird

s

[Win(1) == True, Win(2) == True, Win(3) == True]

In [4]:
# recursion
for i in range(3,21): # adjustment for zero indexing
    s.add(Win[i] == Or(Not(Win[i-1]), Not(Win[i-2]), Not(Win[i-3])))

s.push()
s

[Win(1) == True,
 Win(2) == True,
 Win(3) == True,
 Win(4) == Or(Not(Win(3)), Not(Win(2)), Not(Win(1))),
 Win(5) == Or(Not(Win(4)), Not(Win(3)), Not(Win(2))),
 Win(6) == Or(Not(Win(5)), Not(Win(4)), Not(Win(3))),
 Win(7) == Or(Not(Win(6)), Not(Win(5)), Not(Win(4))),
 Win(8) == Or(Not(Win(7)), Not(Win(6)), Not(Win(5))),
 Win(9) == Or(Not(Win(8)), Not(Win(7)), Not(Win(6))),
 Win(10) == Or(Not(Win(9)), Not(Win(8)), Not(Win(7))),
 Win(11) == Or(Not(Win(10)), Not(Win(9)), Not(Win(8))),
 Win(12) == Or(Not(Win(11)), Not(Win(10)), Not(Win(9))),
 Win(13) == Or(Not(Win(12)), Not(Win(11)), Not(Win(10))),
 Win(14) == Or(Not(Win(13)), Not(Win(12)), Not(Win(11))),
 Win(15) == Or(Not(Win(14)), Not(Win(13)), Not(Win(12))),
 Win(16) == Or(Not(Win(15)), Not(Win(14)), Not(Win(13))),
 Win(17) == Or(Not(Win(16)), Not(Win(15)), Not(Win(14))),
 Win(18) == Or(Not(Win(17)), Not(Win(16)), Not(Win(15))),
 Win(19) == Or(Not(Win(18)), Not(Win(17)), Not(Win(16))),
 Win(20) == Or(Not(Win(19)), Not(Win(18)), Not(Win(17))),
 Win(21) == Or(Not(Win(20)), Not(Win(19)), Not(Win(18)))]

- Finally, we check if the first player can win, knowing that they always start with a pile size of 21.
    - We want to show that the above setup leads to the conclusion $Win(21)$.
    - We add the negation of the conclusion to the premises and show that it is UNSAT.

In [5]:
s.add(Win[20] == False) # 20 because indexing

s.check()

unsat

The first player can thus always win. What about the winning strategy?

- Let's see if we can infer something by pushing $Win(21) = \top$ and getting the correponding valualtion (model)

In [6]:
s.pop()

s.add(Win[20] == True)

display(s.check())
s.model()

sat

[Win(1) = True,
 Win(4) = False,
 Win(9) = True,
 Win(17) = True,
 Win(13) = True,
 Win(20) = False,
 Win(14) = True,
 Win(15) = True,
 Win(21) = True,
 Win(3) = True,
 Win(16) = False,
 Win(18) = True,
 Win(19) = True,
 Win(5) = True,
 Win(8) = False,
 Win(10) = True,
 Win(6) = True,
 Win(11) = True,
 Win(12) = False,
 Win(7) = True,
 Win(2) = True]

- Let's observe the model more carefully:

In [7]:
for i in range(20,-1,-1):
    display(Markdown(f"For pile of size = {i+1}, can we make a winning choice, $ Win({i+1})$? $ \\ \\ {s.model()[Win[i]]}$"))

For pile of size = 21, can we make a winning choice, $ Win(21)$? $ \ \ True$

For pile of size = 20, can we make a winning choice, $ Win(20)$? $ \ \ False$

For pile of size = 19, can we make a winning choice, $ Win(19)$? $ \ \ True$

For pile of size = 18, can we make a winning choice, $ Win(18)$? $ \ \ True$

For pile of size = 17, can we make a winning choice, $ Win(17)$? $ \ \ True$

...

For pile of size = 5, can we make a winning choice, $ Win(5)$? $ \ \ True$

For pile of size = 4, can we make a winning choice, $ Win(4)$? $ \ \ False$

For pile of size = 3, can we make a winning choice, $ Win(3)$? $ \ \ True$

For pile of size = 2, can we make a winning choice, $ Win(2)$? $ \ \ True$

For pile of size = 1, can we make a winning choice, $ Win(1)$? $ \ \ True$

- Note that every multiple of 4 is false. This is precisesly the winning strategy. 


    - The first player can always force the second player into a multiple of 4. 
    - The second player can never force the first player into a multiple of 4.
    - Assuming optimal play by the first player, eventually, the second player will be forced into a state with a file size of 4.
        - They cannot remove all object from the pile here, so they remove 1, 2, or 3 onjects. 
        - Whatever their choice, the first player ends up in cases they necessarily win!

### **Non Linear Constraint Solving**

This is pretty straightforward, we just load the statements into a z3 solver.

In [8]:
x, y = Ints('x y')

In [9]:
type(x)

z3.z3.ArithRef

In [10]:
s = Solver()

In [11]:
s.add(
    x*x + y*y == 25,  # pyright: ignore[reportOperatorIssue]
    x + y == 7,
    x > 0,
    y > 0
)

s

[x*x + y*y == 25, x + y == 7, x > 0, y > 0]

In [12]:
s.check()

sat

While SAT, iterate append negation of solution to find new assignment, till UNSAT is reached. Display each solution as it comes.

In [13]:
while str(s.check()) != 'unsat':

    m = s.model()

    sol_x = int(str(m[x]))
    sol_y = int(str(m[y]))

    print("x =", sol_x, ", y =", sol_y)
    s.add(Not(And(x == sol_x, y == sol_y)))

x = 3 , y = 4
x = 4 , y = 3


$(x=3, y=4)$ and $(x=4,y=3)$ are the only integer solutions.

### **(Inductive) Invariant Synthesis Example**

- At the time of writing - it's almost the end of the semester and I'm running short on time. So I'm going to outline an approach that we'll reuse in Week 6 (and subsequently) for invariant synthesis
 
- We'll implement Farkas' Lemma as described in [\[CVA03.pdf\]](..\week8\papers\CAV03.pdf) - for any d variables in a program. In the invariant, these variables may be products of each other. Let's formulate mathematically.

We have program variables $v = \begin{bmatrix}x_1 \\ x_2 \\ \vdots \\ x_d \end{bmatrix}$
We construct the following basis over the program variables:

$$
b(v) = \begin{bmatrix}1 \\ x_1 \\ x_2 \\ \vdots \\ x_d \\ x_1x_2 \\ \vdots \\ x_2^2 \\ \vdots \\ x_1^3 \\ \vdots \\ x_1x_2x_3...x_d \\ \vdots \\ x_d^d\end{bmatrix}
$$

Our Linear\* Invariant(s) will be over this basis. Consider the coefficients:

$$c = \begin{bmatrix}a_1 \\ a_2 \\ \vdots \\ a_k\end{bmatrix}$$

\*These are linear not in $v$ but in $b(v)$. Also, $k$ is at most $\binom{2d}{d}$

The expression for our invariant inequality becomes

$$I(b(v)) = c^{\top} \times b(v) \leq 0$$

For our base case, let $v_{init}$ be the initial state of the loop variables, $b(v)_{init}$ is the initial state of the basis variables. We ensure that $I(b(v)_{init}) \leq 0$


Now, we have three types of constraints:

1. Assignment Constraints 

    `x := x+1`

2. Condition Constraints 

    `while (i<n)`

3. Loop Body Constraints 

    `if (x > 0) {y := y + 1;}`

Note that 1 and 2 can be combined to give something like 3, which holds within the loop. We can thus write any expression in the loop in this format. 

In other words, a given path through a loop thus contains conditions or 'guards' on the variables, $G(v)$, and assignments or transitions on the variables, $T(v)$. We write, $\forall \ b(v)$:

$$
I(b(v)) \land G(b(v)) \implies I(b(T(v)))
$$

Let's formalise $G(b(v))$. Suppose we have some execution path through the loop. We set some guard coefficients

$$
g = \begin{bmatrix}g_1 \\ g_2 \\ \vdots \\ g_k \end{bmatrix}
$$

And note that

$$
G(b(v)) = g^\top \times b(v) \leq 0
$$

This inequality on the basis describes the while loop condition and the respective execution path it takes through if conditions (if present).

Similarly, let's formalise $b(T(v))$. $T(v)$ defines a vector containing the transition updates of every variable. We update everything in the basis with its transition value and expand out.

This gives us $b(T(v))$, a new basis with linear combination of the original basis elements at each entry.

We know that this can be represented with a matrix multiplication! Consider a $k \times k$ square matrix $M$ where each row contains the coefficients of the transition state for the corresponding entry in the original basis. 

There are k columns, each with the coefficient of a particular basis element in the transition state of that entry.

To help visualise - suppose the $i$ th entry of $T(v)$, $T(v)_i$, is given by $t_0x_1 + t_1x_2 + \dots + t_dx_d$, i.e. the transition update for variable $x_i$. Note that this is always linear in $v$ - see next point for why.

Applying these transitions to some $b(v)_j$, we expand out the terms and end up with a new linear combination over the variables of $b(v)$ This is $b(T(v))_j = \begin{bmatrix}b_{j1} & b_{j2} & b_{j3} & \dots & b_{jk} \end{bmatrix} b(v)$.

If $T(v)$ was not linear in $v$, than this basis would no longer be a linear combination of the variables in $b(v)$ - it would be a new basis with a new set of terms, which would keep growing for each update!

Anyway, row $j$ of the matrix $M$ will have the column entries $b_{j1}, b_{j2} , b_{j3} , \dots , b_{jk}$.

We can thus write:

$$
b(T(v)) = M \times b(v)
$$

(This will be clearer with an example later on.)

Hence, we can derive the following:

$$
I(b(v)) \land G(b(v)) \implies I(b(T(v)))
$$

$$
(c^\top b(v) \leq 0) \land (g^\top b(v) \leq 0) \implies (c^\top  M b(v) \leq 0)
$$

By [Farkas' Lemma](https://www.mat.univie.ac.at/~rabot/publications/jour05-01.pdf), this implication holds **iff** $\exists \ \lambda_0,\lambda_1 \geq 0$ such that:

$$
c^\top  M b(v) = \lambda_0 c^\top b(v) + \lambda_1 g^\top b(v)
$$

Now,

$$
c^\top  M b(v) = \lambda_0 c^\top b(v) + \lambda_1 g^\top b(v)
$$

$$
\implies (c^\top  M b(v))^\top = (\lambda_0 c^\top b(v))^\top + (\lambda_1 g^\top b(v))^\top
$$

$$
\implies (c^\top  M b(v))^\top = \lambda_0 (c^\top b(v))^\top + \lambda_1 (g^\top b(v))^\top
$$

$$
\implies b(v)^\top M^\top c = \lambda_0 b(v)^\top c  + \lambda_1 b(v)^\top g 
$$

$$
\implies b(v)^\top M^\top c =  b(v)^\top \lambda_0 c  +  b(v)^\top \lambda_1 g 
$$

$$
\implies M^\top c = \lambda_0  c +  \lambda_1 g 
$$

We just reduced our inductive invariant to a system of equations, which, as noted in the previous section, can be solved using Non-Linear Constraint Solving in Z3! 

Our unknowns are the real coefficients $c$, and scalars $\lambda_0,\lambda_1$. The scalars must be $\geq 0$. We also add the constraint that $c^\top c = 1$ to normalize the coefficients - this ensures that the solver doesn't keep coming up with scalar multiples of the same expression.

It's time to run through this using `i` and `s` (`s` representing `sum`)

- Setup the Coefficient Vector $c$

In [14]:
c = np.array((Reals('a1 a2 a3 a4 a5 a6')))

print(c)

[a1 a2 a3 a4 a5 a6]


- Ensure that all solutions are normalised - this prevents trivial solutions and solutions which are scalar multiples of each other

In [15]:
norm_constraint = c.T @ c == 1
norm_constraint

a1*a1 + a2*a2 + a3*a3 + a4*a4 + a5*a5 + a6*a6 == 1

- Initial Constraint (base case), and limit for parameter

In [16]:
i = 0
s = 0
n = 2**7-1 # parameter limit

v = [i, s]
b_v_init = [1, i, s, i*i, i*s, s*s]

init_constraint = c.T @ b_v_init <= 0

init_constraint

a1*1 + a2*0 + a3*0 + a4*0 + a5*0 + a6*0 <= 0

- Transition Matrix

In [17]:
M = np.array([
        [1, 0, 0, 0, 0, 0], # 1 = 1
        [1, 1, 0, 0, 0, 0], # i = i+1 = 1*1 + 1*i + 0*(others)
        [0, 1, 1, 0, 0, 0], # s = s+i = 0*1 + 1*i + 1*s ...etc
        [1, 2, 0, 1, 0, 0], # (i+1)^2 = i^2 + 2i + 1
        [0, 1, 1, 1, 1, 0], # (i+1)(s+i) = i + s + i^2 + i*s
        [0, 0, 0, 1, 2, 1], # (s+i)^2 = i^2 + 2*i*s + s^2 
    ])

print(M.T) # transpose

[[1 1 0 1 0 0]
 [0 1 1 2 1 0]
 [0 0 1 0 1 0]
 [0 0 0 1 1 1]
 [0 0 0 0 1 2]
 [0 0 0 0 0 1]]


- Guard Vector

In [18]:
g = np.array([1-n, 1, 0, 0, 0, 0])# i < n ===> i <= n-1 ===> 1*(1-n) + 1*i + 0*(others)

print(g)

[-126    1    0    0    0    0]


- Farkas Scalars

In [19]:
λ0, λ1 = Reals('λ0 λ1')

- Setup equations

In [20]:
lhs = M.T @ c
rhs = c*λ0 + g*λ1

print(lhs)
print()
print(rhs)

[1*a1 + 1*a2 + 0*a3 + 1*a4 + 0*a5 + 0*a6
 0*a1 + 1*a2 + 1*a3 + 2*a4 + 1*a5 + 0*a6
 0*a1 + 0*a2 + 1*a3 + 0*a4 + 1*a5 + 0*a6
 0*a1 + 0*a2 + 0*a3 + 1*a4 + 1*a5 + 1*a6
 0*a1 + 0*a2 + 0*a3 + 0*a4 + 1*a5 + 2*a6
 0*a1 + 0*a2 + 0*a3 + 0*a4 + 0*a5 + 1*a6]

[a1*λ0 + -126*λ1 a2*λ0 + 1*λ1 a3*λ0 + 0*λ1 a4*λ0 + 0*λ1 a5*λ0 + 0*λ1
 a6*λ0 + 0*λ1]


In [21]:
farkas_constraints = [lhs[i] == rhs[i] for i in range(len(c))]

farkas_constraints

[1*a1 + 1*a2 + 0*a3 + 1*a4 + 0*a5 + 0*a6 == a1*λ0 + -126*λ1,
 0*a1 + 1*a2 + 1*a3 + 2*a4 + 1*a5 + 0*a6 == a2*λ0 + 1*λ1,
 0*a1 + 0*a2 + 1*a3 + 0*a4 + 1*a5 + 0*a6 == a3*λ0 + 0*λ1,
 0*a1 + 0*a2 + 0*a3 + 1*a4 + 1*a5 + 1*a6 == a4*λ0 + 0*λ1,
 0*a1 + 0*a2 + 0*a3 + 0*a4 + 1*a5 + 2*a6 == a5*λ0 + 0*λ1,
 0*a1 + 0*a2 + 0*a3 + 0*a4 + 0*a5 + 1*a6 == a6*λ0 + 0*λ1]

- Insert Constraints into Z3 Solver

In [22]:
sol = Solver()
sol.add(norm_constraint, init_constraint, λ0 >= 0, λ1 >= 0)
sol.add(farkas_constraints)
sol.set('timeout', 5000)

sol

[a1*a1 + a2*a2 + a3*a3 + a4*a4 + a5*a5 + a6*a6 == 1,
 a1*1 + a2*0 + a3*0 + a4*0 + a5*0 + a6*0 <= 0,
 λ0 >= 0,
 λ1 >= 0,
 1*a1 + 1*a2 + 0*a3 + 1*a4 + 0*a5 + 0*a6 == a1*λ0 + -126*λ1,
 0*a1 + 1*a2 + 1*a3 + 2*a4 + 1*a5 + 0*a6 == a2*λ0 + 1*λ1,
 0*a1 + 0*a2 + 1*a3 + 0*a4 + 1*a5 + 0*a6 == a3*λ0 + 0*λ1,
 0*a1 + 0*a2 + 0*a3 + 1*a4 + 1*a5 + 1*a6 == a4*λ0 + 0*λ1,
 0*a1 + 0*a2 + 0*a3 + 0*a4 + 1*a5 + 2*a6 == a5*λ0 + 0*λ1,
 0*a1 + 0*a2 + 0*a3 + 0*a4 + 0*a5 + 1*a6 == a6*λ0 + 0*λ1]

In [23]:
sol.check()

sat

- #### Search for Invariants
The search method deserves some explanation, hence, some digression.

- The solver finds some satisfying coefficients. The initial ones are likely on the trivial side
- To prevent finding invariants that rehash the same result in different ways ($i < n $; $i < n+1$; $i < n+2$; etc), we add a vector dot product requirement which prevents similar invariants from showing up again and again. This will sort of act as a our ranking technique.

    - We take a dot product of the solution vector $c_s$ and the (symbolic) coefficient vector $c$.
    - We ensure that this $c_s^\top c$ is $\leq$ some threshold $t \in [0,1]$ by adding it as a new constraint to the solver.
        - What does this do? $t=1$ means that the vectors $c_s$ and $c$ are identical. By adding this as a constraint to the solver, we are telling it - "don't find the same solution again."
            - Note that we are searching for vectors in a k-dimensional hypersphere (after normalisation) (k=6 in this case). The dot product of the vectors corresponds to the cosine of the angle between them, $\cos \theta$. $t$ thus measures cosine similarity.
            - $t=1 \implies \cos \theta = 1 \implies \theta = 0$, i.e.vectors of equal size (since they are normalised) pointing in the same direction. Hence, identical.
        - If $t=0$, we are saying we want $\theta = 90^\circ$: vectors which are orthogonal. In other words, the next $c_s$ should be completely dissimilar to the ones before, with no components in previous directions. This forces the solver to look in unique solution spaces and find very different invariants.
        - We ideally want something in between - which allows for some degree of refinement ($i <= 100$; followed by $i >= 0$) in existing invariants while exploring different combinations and relations between variables.
            - $t = 0.5$ is what I decided on after some experimentation. This essentially tries to map out solutions in a way that no two found solutions are within $60^\circ$ of each other in the hypersphere.



In [24]:
t = 0.5

curr = time.time()
end = curr + 10

sol.push()
while curr < end:
    if sol.check() == sat:
        m = sol.model()
        c_s = np.array([m[coeff] for coeff in c])
        print("Found Solution:", c_s)
        sol.add(c_s.T @ c <= t)
    # else, the solver simply times out and we try again, 
    # possibly from a different initialization point
    curr = time.time()

sol.pop()


Found Solution: [-0.9999694819? 1/128 0 0 0 0]
Found Solution: [0 0.4082482904? 0.8164965809? -0.4082482904? 0 0]
Found Solution: [0 -1 0 0 0 0]
Found Solution: [0 -0.4082482904? -0.8164965809? 0.4082482904? 0 0]


#### Results

It \[BLEEP\]ing works! Let's analyse the solutions from the latest run (at the time of upload):

```
Found Solution: [-0.9999694819? 1/128 0 0 0 0]
Found Solution: [0 0.4082482904? 0.8164965809? -0.4082482904? 0 0]
Found Solution: [0 -1 0 0 0 0]
Found Solution: [0 -0.4082482904? -0.8164965809? 0.4082482904? 0 0]
```

Let's intepret these.
1. Here, we can clearly see:

$$(-0.999)1 + \left( \frac{1}{128}\right)i + 0 + 0 + 0  + 0 \leq 0$$
$$\approx i \leq 128$$

We would ideally arrive at $i \leq 127 = n$, but this is a good start. The solving loop will find this too, if the threshold is relaxed to 1 (but that opens up the search space too much and z3 keeps coming up with stupid invariants, hence, we stick with this).

2. This one is crazy! It's part of the a invariant! We have:
$$0 + 0.408i + 0.816s + -0.408i^2 + 0 + 0 \leq 0$$

... which is essentially the normalised version of:
$$i + 2s - i^2 \leq 0$$
$$2s \leq i^2 - i$$
$$s \leq \frac{i(i-1)}{2}$$

3. Pretty simple, but also crucial. The solve learns that $-i \leq 0 \implies i >= 0$.

4. This is almost identical to 2, but flips some signs. We essentially have the normalised version of:
$$-i - 2s + i^2 \leq 0$$
$$2s \geq i^2 - i$$
$$s \geq \frac{i(i-1)}{2}$$

AND-ing these together, for $n=127$, we arrive that the invariants:

$$0 \leq i \leq 128; \hspace{0.5cm} s = \frac{i(i-1)}{2}$$

Which is pretty good! Different values of $n$ lead to different upper bounds on $i$ in the first invariant. Z3 seems to begin its search with powers of two, so experimental verification on the exact bound on n would be useful.


#### Experiments

We can stop here, but I want to make this robust enough for the rest of the implementation in the upcoming weeks. I want to be able to find $i \leq n$ as an invariant, and more generally, proper termination invariants. If this was a Dafny program, that's the only thing missing which prevents us from finding all the invariants for sum and directly reinserting them.

The main issue is that the parameter $n$ is being passed as constant value. We should instead include the parameter (and its products with the other variables) in the basis. For now, let's just include $n$.

- Setup and Corrections

Using $n$ symbolically and adding an entry for it in $M$, $c$ and $g$ alone is problematic. This keeps $n$ as a free variable. This means for a z3 solver finding assignments for coefficients, finds the easiest assignment for $n$ it sees. In certain experiments, this leads to it finding $n \geq 0$ correctly, but then $n \leq 0$ as well, because $n=0$ (set freely) acts as a valid solution for both. This is wrong, as the solver thus ends up deriving $n=0$ as the inequality.

Thus, it makes sense to introduce something like:

$$\forall n \ ( n \geq 0 \implies c ^\top b(v)_{init} \leq 0) \equiv \forall n \ ( -n \leq 0 \implies c ^\top b(v)_{init} \leq 0)$$

More generally, for all parameters, the precondition implies initialization. We have access to both in this problem, and in Dafny.

However, Z3 doesn't play nice at ALL with universal quantification, and mixing it in here led us to massive slowdowns.

Note that we can address this issue the same way we did for the loop - Farkas' Lemma!

$$\forall n \ ( -n \leq 0 \implies c ^\top b(v)_{init} \leq 0)$$
$$ \equiv $$
$$ \exists \ \lambda_{p_0}, \lambda_{p_1} \left( c ^\top b(v)_{init} =  \lambda_{p_0}(-1) + \lambda_{p_1}(-n) \right)$$

But hold on - where did that $(-1)$ term come from? Well, it comes from a trivial truth we know about real numbers: $-1 \leq 0$. And so, we actually used:

$$\forall n \ \left( (-n \leq 0) \land (-1 \leq 0)  \implies c ^\top b(v)_{init} \leq 0 \right)$$

To get 
$$ c ^\top b(v)_{init} =  \lambda_{p_0}(-1) + \lambda_{p_1}(-n)$$
Unfortunately, Z3 doesn't entirely play nice with this either - because on simplification, we end up with an $n$ term on both sides which we can cancel - but Z3 can't. It ends up setting $n$ as a free variable. However, fixing this isn't hard - it becomes a job of matching the sum of the constant-coefficient elements of $c ^\top b(v)_{init}$ (by setting the $n=0$) with the $-\lambda_{p_0}$, and similarly, for the element of $c$ representing the coefficient of $n$, pair it with $-\lambda_{p_1}$.

To generalise this for multiple preconditions, we can simply add a new precondition to the right side by taking some $\lambda_i$ times the LHS term of the precondition inequality ($\leq$), when the RHS is set to $0$. Then, proceed with coefficient matching.


We actually should be correcting this in our primary Farkas' derivation from earlier as well. 

$$
c^\top  M b(v) = \lambda_{i_0} c^\top b(v) + \lambda_{i_1} g^\top b(v) + \lambda_{i_2}t^\top b(v)
$$

Where $t$ is a trivial vector given by $\begin{bmatrix}-1 \\0 \\0 \\. \\. \\ 0 \end{bmatrix}$. Proceeding with the derivation as we did earlier, we arrive at:

$$
M^\top c = \lambda_{i_0}  c +  \lambda_{i_1} g + \lambda_{i_2} t
$$

Note - this is technically the proper correct version of Farkas, and our older one was functional only for constant $n$. The fact that it figured out $i \geq 0$ is because of the guard, and through the use of a constant $n$. It's essentially matching coefficients in the transition equation $-i - 1 = \lambda_0(-i) + \lambda_1(i - 126)$, where it's able to find $\lambda_1 = \frac{1}{126} \geq 0$, and using that, $\lambda_0 = \frac{127}{126} \geq 0$. However, with a symbolic $n$, we solve $-i - 1 = \lambda_0(-i) + \lambda_1(i - n + 1)$. Due to the lack of an $n$ term in $i \geq 0$, $\lambda_1 =0$, but if we simplify for the constant term, $\lambda_1 =-1$, leading to a contradiction. With the update, we have $-i - 1 = \lambda_{i_0}(-i) + \lambda_{i_1}(i - n + 1) + \lambda_{i_2}(-1)$, which is able to find nonnegative values for all the $\lambda s$  and thus is able to prove $i \geq 0$.

There is, however, still an issue. While it's now logically correct, this new setup has a weakness. The abscenece of the trivial term essentially made sure that certain inequalites always found strict results. Consider the invariant $a_2i + 2s - i^2 \leq 0$. This has the transition $a_2(i+1) + (2s + 2i) - (i^2 + 2i + 1) \leq 0$. In other words, this gives $a_2i + 2s - i^2 + (a_2 -1) \leq 0$. Now, as we did before, we match the coefficients to find if proper $\lambda s$ exist. We note:  $a_2i + 2s - i^2 + (a_2 - 1)= \lambda_{i_0}(a_2i + 2s - i^2) + \lambda_{i_1}(i - n + 1) + \lambda_{i_2}(-1)$. As before, $\lambda_{i_1} = 0$ due to the lack of an $n$ term, and clearly$\lambda_{i_0} = 1$. We're thus left with $(a_2 - 1) \lambda_{i_2}(-1)$. Since $\lambda_{i_2} \geq 0$, this means we find $a_2$ such that $(a_2 - 1) \leq 0$ or $a_2 \leq 1$. This means it can find the exact inequality we're looking for = $a_2 = 1$, or anything smaller than that, which still satisfies the inequality but is weaker. However - if $\lambda_{i_2}$ absent, it's forced to set $(a_2 - 1) = 0$ and finds $a_2 = 1$ directly. This is why the old method found the strict inequalities quickly. But with this new, relaxed method, we may not find it!

To fix this, we simply ask the solver to find successive solutions which minimise $\lambda_{i_2}$. This, combined with our vectorised blocking logic should work. 

(The blocking helps in case there is a chance where incrementally improving $\lambda_{i_2}$ lets us approach the local minima of a perfect solution - that has likely already been found - but never truly reach it, just hovering around. Vectorised blocking stops the rapid approach.)

In [33]:
i = 0
s = 0
n = Real('n')

v = [i, s]
b_v_init = [1, i, s, i*i, i*s, s*s, n]
b_v_init_consts = [1, i, s, i*i, i*s, s*s, 0]

k = len(b_v_init)
indices = range(k)

c = np.array((Reals('a1 a2 a3 a4 a5 a6 a7')))
norm_constraint = c.T @ c == 1

norm_constraint

a1*a1 + a2*a2 + a3*a3 + a4*a4 + a5*a5 + a6*a6 + a7*a7 == 1

In [34]:
λp0, λp1 = Reals('λp0 λp1')
#precondition_constraint = ForAll([n], Implies(n >= 0, c.T @ b_v_init <= 0))
#precondition_constraint = c.T @ b_v_init == λp0 *(-n) + λp1 * (-1)
precondition_constraint = [c.T @ b_v_init_consts == -λp0, c[6] == -λp1] 


precondition_constraint

[a1*1 + a2*0 + a3*0 + a4*0 + a5*0 + a6*0 + a7*0 == -λp0, a7 == -λp1]

In [35]:
M = np.array([
        [1, 0, 0, 0, 0, 0, 0], # 1 = 1
        [1, 1, 0, 0, 0, 0, 0], # i = i+1 = 1*1 + 1*i + 0*(others)
        [0, 1, 1, 0, 0, 0, 0], # s = s+i = 0*1 + 1*i + 1*s ...etc
        [1, 2, 0, 1, 0, 0, 0], # (i+1)^2 = i^2 + 2i + 1
        [0, 1, 1, 1, 1, 0, 0], # (i+1)(s+i) = i + s + i^2 + i*s
        [0, 0, 0, 1, 2, 1, 0], # (s+i)^2 = i^2 + 2*i*s + s^2 
        [0, 0, 0, 0, 0, 0, 1], # n = n
    ])

λi0, λi1, λi2 = Reals('λi0 λi1 λi2')
g = np.array([1, 1, 0, 0, 0, 0, -1])
t = np.array([-1, 0, 0, 0, 0, 0, 0])
lhs = M.T @ c
rhs = c*λi0 + g*λi1 + t*λi2
farkas_constraints = [lhs[i] == rhs[i]  for i in indices]

farkas_constraints

[1*a1 + 1*a2 + 0*a3 + 1*a4 + 0*a5 + 0*a6 + 0*a7 ==
 a1*λi0 + 1*λi1 + -1*λi2,
 0*a1 + 1*a2 + 1*a3 + 2*a4 + 1*a5 + 0*a6 + 0*a7 ==
 a2*λi0 + 1*λi1 + 0*λi2,
 0*a1 + 0*a2 + 1*a3 + 0*a4 + 1*a5 + 0*a6 + 0*a7 ==
 a3*λi0 + 0*λi1 + 0*λi2,
 0*a1 + 0*a2 + 0*a3 + 1*a4 + 1*a5 + 1*a6 + 0*a7 ==
 a4*λi0 + 0*λi1 + 0*λi2,
 0*a1 + 0*a2 + 0*a3 + 0*a4 + 1*a5 + 2*a6 + 0*a7 ==
 a5*λi0 + 0*λi1 + 0*λi2,
 0*a1 + 0*a2 + 0*a3 + 0*a4 + 0*a5 + 1*a6 + 0*a7 ==
 a6*λi0 + 0*λi1 + 0*λi2,
 0*a1 + 0*a2 + 0*a3 + 0*a4 + 0*a5 + 0*a6 + 1*a7 ==
 a7*λi0 + -1*λi1 + 0*λi2]

In [36]:
sol = Solver()

sol.add(norm_constraint)
sol.add(precondition_constraint)
sol.add(λp0 >= 0, λp1 >= 0, λi0 >= 0, λi1 >= 0, λi2 >= 0)
sol.add(farkas_constraints)

sol.set('timeout', 100)

In [37]:
curr = time.time()
end = curr + 0.5

sol.push()
sol.add(c[0]==0, c[4]==0, c[5]==0, c[6]==0)
while curr < end:
    if sol.check() == sat:
        m = sol.model()
        c_s = np.array([m[coeff] for coeff in c])
        print("Found Solution:", c_s)

        λi2_s = m[λi2]
        sol.add(λi2 <= λi2_s - 0.01)
        sol.add(c_s.T @ c <= 0.866)

    curr = time.time()

sol.pop()

Found Solution: [0 -0.8291561975? 1/2 -1/4 0 0 0]
Found Solution: [0 0.4082482904? 0.8164965809? -0.4082482904? 0 0 0]


- Update to solving strategy
    We now start by zeroing out invariants and incrementally exploring the solution space, we got quite lucky last time when we didnt do this.

In [38]:
ε = 0.01
t = 0.866

sol.push()

found = set()
for choice_count in range(1, k+1):
    for active_coeffs in combinations(indices, choice_count):

        #print("Active:", [c[i] for i in active_coeffs])
        inactive_coeffs = [c[i] == 0 for i in indices if i not in active_coeffs]
        sol.push()
        sol.add(inactive_coeffs)

        curr = time.time()
        end = curr + 1

        while curr < end:
            if sol.check() == sat:
                m = sol.model()
                c_s = np.array([m[coeff] for coeff in c])

                if str(c_s) not in found:
                    print("Found Solution:", c_s)
                    found.add(str(c_s))
                
                λi2_s = m[λi2]
                sol.add(λi2 <= λi2_s - ε)
                sol.add(c_s.T @ c <= t)

            curr = time.time()

        sol.pop()

sol.pop()

Found Solution: [-1 0 0 0 0 0 0]
Found Solution: [0 -1 0 0 0 0 0]
Found Solution: [0 0 0 0 0 0 -1]
Found Solution: [-1/2 -0.8660254037? 0 0 0 0 0]
Found Solution: [0 0.7071067811? 0 0 0 0 -0.7071067811?]
Found Solution: [0 0 0.8944271909? -0.4472135954? 0 0 0]
Found Solution: [0 -0.8291561975? 1/2 -1/4 0 0 0]
Found Solution: [0 0.4082482904? 0.8164965809? -0.4082482904? 0 0 0]
Found Solution: [0 0 1/2 -1/4 0 0 -0.8291561975?]
Found Solution: [-1/2 -0.6614378277? 1/2 -1/4 0 0 0]
Found Solution: [-0.8660254037? -1/2 0 0 0 0 0]
Found Solution: [0 1/4 0.8660254037? -0.4330127018? 0 0 0]
Found Solution: [0 -0.4082482904? -0.8164965809? 0.4082482904? 0 0 0]
Found Solution: [0 0.2072890493? 7/8 -7/16 0 0 0]
Found Solution: [0 -1/2 -1/2 1/4 0 0 -0.6614378277?]
Found Solution: [0 -1/2 0 0 0 0 -0.8660254037?]
Found Solution: [-0.8291561975? 0 1/2 -1/4 0 0 0]
Found Solution: [-0.8196798155? 1/8 1/2 -1/4 0 0 0]
Found Solution: [-0.6614378277? -1/2 -1/2 1/4 0 0 0]
Found Solution: [0 1/8 1/2 -1/4 0 

It absolutely works. We find all the necessary invariants to prove this in Dafny. Here's a copy a trace:

```
Found Solution: [-1 0 0 0 0 0 0]
Found Solution: [0 -1 0 0 0 0 0]
Found Solution: [0 0 0 0 0 0 -1]
Found Solution: [-1/2 -0.8660254037? 0 0 0 0 0]
Found Solution: [0 0.7071067811? 0 0 0 0 -0.7071067811?]
Found Solution: [0 0 0.8944271909? -0.4472135954? 0 0 0]
Found Solution: [-0.1410673597? -99/100 0 0 0 0 0]
Found Solution: [-0.6614378277? -3/4 0 0 0 0 0]
Found Solution: [0 1/4 0.8660254037? -0.4330127018? 0 0 0]
Found Solution: [0 -0.4082482904? -0.8164965809? 0.4082482904? 0 0 0]
Found Solution: [0 0 1/2 -1/4 0 0 -0.8291561975?]
Found Solution: [-0.8660254037? -1/2 0 0 0 0 0]
Found Solution: [-1/2 -1/2 0 0 0 0 -0.7071067811?]
Found Solution: [0 -0.8291561975? -1/2 1/4 0 0 0]
Found Solution: [0 0.4082482904? 0.8164965809? -0.4082482904? 0 0 0]
Found Solution: [-0.8291561975? 0 1/2 -1/4 0 0 0]
Found Solution: [0 -1/2 -1/2 1/4 0 0 -0.6614378277?]
Found Solution: [-0.6614378277? -1/2 -1/2 1/4 0 0 0]
```

The key solutions are: `[0 -1 0 0 0 0 0]`, `[0 0.7071067811? 0 0 0 0 -0.7071067811?]`, `[0 -0.4082482904? -0.8164965809? 0.4082482904? 0 0 0]`, `[0 0.4082482904? 0.8164965809? -0.4082482904? 0 0 0]`
